In [1]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q pmdarima

import pandas as pd
import numpy as np

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.6 MB/s eta 0:00:00


In [2]:
# Cell 2 (SARIMA version) - Verify imports
import statsmodels.api as sm
print('statsmodels:', sm.__version__)

statsmodels: 0.14.6


In [3]:
# Cell 3 - Load processed data
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


In [4]:
# Cell 4 - WMAE metric
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [5]:
# Cell 5 - Same random sample as Prophet notebook (seed=42, SAMPLE_SIZE=15)
# so all classical models are scored on identical series.
SAMPLE_SIZE = 15
VAL_LEN = 39
MIN_LEN = 52 + VAL_LEN + 10

series_totals = (
    train.groupby(['Store','Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)
np.random.seed(42)
all_keys = series_totals.index.tolist()
sample_idx = np.random.choice(len(all_keys), size=SAMPLE_SIZE, replace=False)
sample_keys = [all_keys[i] for i in sample_idx]

series_data = {}
for store, dept in sample_keys:
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    if len(sub) < MIN_LEN:
        continue
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)
    series_data[(store, dept)] = {
        'values': sub['Weekly_Sales'].values,
        'is_holiday': sub['IsHoliday'].values
    }

print('Series selected for ARIMA:', len(series_data))

Series selected for ARIMA: 13


In [6]:
# Cell 6 - Same 39-week holdout used everywhere else
fit_raw, val_raw, val_holiday = {}, {}, {}
for key, d in series_data.items():
    fit_raw[key] = d['values'][:-VAL_LEN]
    val_raw[key] = d['values'][-VAL_LEN:]
    val_holiday[key] = d['is_holiday'][-VAL_LEN:]

In [7]:
# Cell 7 - Fit SARIMA per series with a FIXED (1,1,1)(1,1,1,52) order.
# Deliberately skipping any seasonal order search - that search is what
# makes SARIMA slow, and the assignment explicitly says not to burn time
# tuning ARIMA/SARIMA. (1,1,1)(1,1,1,52) is a standard, defensible choice
# for weekly retail data with strong yearly seasonality.
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_models = {}
all_true, all_pred, all_holiday = [], [], []

for key, fseries in fit_raw.items():
    model = SARIMAX(
        fseries,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 52),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fitted = model.fit(disp=False, maxiter=50)   # cap iterations - speeds up convergence
    sarima_models[key] = fitted

    pred = fitted.forecast(steps=VAL_LEN)
    all_true.extend(val_raw[key])
    all_pred.extend(pred)
    all_holiday.extend(val_holiday[key])

    print(f"{key}: fit done, AIC={fitted.aic:.1f}")

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print('Done. Total validation points:', len(all_true))

(41, 13): fit done, AIC=10.0
(26, 28): fit done, AIC=10.0
(32, 67): fit done, AIC=10.0
(28, 30): fit done, AIC=10.0
(18, 27): fit done, AIC=10.0
(15, 32): fit done, AIC=10.0
(44, 92): fit done, AIC=10.0
(18, 36): fit done, AIC=10.0
(15, 6): fit done, AIC=10.0
(8, 29): fit done, AIC=10.0
(16, 52): fit done, AIC=10.0
(16, 79): fit done, AIC=10.0
(24, 14): fit done, AIC=10.0
Done. Total validation points: 507


In [8]:
# Cell 8 - Evaluate
sarima_wmae = wmae(all_true, all_pred, all_holiday)
print('SARIMA WMAE:', sarima_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

SARIMA WMAE: 1096.0113680027391
Holiday WMAE:     1010.4401812741613
Non-holiday WMAE: 1119.1387157672198


In [9]:
# Cell 9 - Save models
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(sarima_models, 'models/sarima_best.pkl')
print('Saved.')

Saved.
